In [ ]:
!pip -q install tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#GROUP1 = ["pigs051119","pigs061119","pigs071119","pigs081119","pigs091119",
#          "pigs151119","pigs161119","pigs271119","pigs281119"]                 # Train("pigs051119","pigs061119","pigs071119","pigs081119")
#GROUP2 = ["pigs291119","pigs301119","pigs011219","pigs021219a","pigs021219b"]  # Val
#GROUP3 = ["pigs031219","pigs041219","pigs051219","pigs071219","pigs081219",
#          "pigs091219","pigs101219a","pigs101219b","pigs111219"]               # Test("pigs051219","pigs071219")

#GROUP_DAYS = ["pigs271119","pigs281119", "pigs291119","pigs301119"]
#SPLIT_NAME = "frame_attribute_(3)"
#GROUP_DAYS = ["pigs271119","pigs281119", "pigs291119","pigs301119"]
#SPLIT_NAME = "frame_attribute_(4)"
#GROUP_DAYS = ["pigs101219a","pigs101219b","pigs111219"]
#SPLIT_NAME = "frame_attribute_(5)"

In [ ]:
from pathlib import Path

BASE_UNZIPPED    = Path("/content/drive/MyDrive/pig_data_unzipped")
GLOBAL_MASK_PATH = BASE_UNZIPPED / "mask.png"         # binary (white=pen)

GROUP_DAYS = ["pigs101219a","pigs101219b","pigs111219"]
SPLIT_NAME = "frame_attribute_(5)"

# ====== ROI ======
ROI_PATH = Path("/content/_annotations.coco.json")

# ====== OUTPUT ======
OUT_ROOT        = Path(f"/content/drive/MyDrive/pig-selected_{SPLIT_NAME}")
CANDIDATES_CSV  = OUT_ROOT / "bursts_candidates.csv"      # Pha 1
OUT_IMG_DIR     = OUT_ROOT / f"images_{SPLIT_NAME}"       # Pha 2
MANIFEST        = OUT_ROOT / f"manifest_{SPLIT_NAME}.csv" # Pha 2
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

# ====== FPS & timestamp ======
FPS_FALLBACK = 6.0
USE_TIME_TXT = True

# ====== BURST ======
BURST_LEN            = 6
BURST_STRIDE_FRAMES  = 3
BURST_OFFSETS        = [-3,-2,-1,0,1,2]

MIN_GAP_BURST_S       = 10.0
TOPK_PER_SEC          = 2
ASPECT_DELTA_TH       = 0.25
SPEED_PEAK_WIN        = 6
BLUR_MIN_VARLAP       = 80.0
ROI_DILATE_PX         = 8
CAP_BURST_PER_MIN_PER_TYPE = {
    "roi_enter": 2,
    "roi_exit" : 2,
    "onset"    : 2,
    "offset"   : 2,
    "speed_peak": 2,
}
CAP_BURST_PER_MIN_TOTAL = 1

SPEED_RMS_LOW    = 2.0
FG_AREA_MIN      = 0.004
ASPECT_DELTA_LOW = 0.05
CAP_SLEEP_PER_MINUTE = 0
SLEEPY_KEEP_QUOTA    = 10

USE_LOCAL_BG  = True
LOCAL_BG_NAME = "background.png"

TARGET_IMAGES      = 3600
SAMPLER_MODE       = "balanced"  # "balanced" | "random"
MAX_SLEEP_RATIO    = 0.25
SLEEP_STRICT       = True
SLEEP_POLICY       = "cap"
RANDOM_SEED        = 42
BATCH_BURSTS       = None

P1_MIN_GAP_GLOBAL_S  = 12.0
P1_HASH_WINDOW_S     = 30.0
P1_AHAMMING_SKIP     = 5

P2_MIN_GAP_GLOBAL_S  = 12.0
P2_HASH_WINDOW_S     = 30.0
P2_AHAMMING_SKIP     = 5

print("OUT_ROOT:", OUT_ROOT)


In [ ]:
import cv2, json, csv, random, os, math, hashlib
import numpy as np
import pandas as pd
from collections import deque, defaultdict
from tqdm import tqdm
from pathlib import Path

def is_depth_video(path: Path) -> bool:
    name = path.name.lower()
    return ("depth" in name) or (name.endswith("_depth.mp4")) or ("_d" in name and name.endswith(".mp4"))

def list_color_videos():
    vids=[]
    for day in GROUP_DAYS:
        dd = BASE_UNZIPPED/day
        if not dd.exists():
            print("[WARN] missing day:", dd);
            continue
        for p in sorted(dd.rglob("*.mp4")):
            if is_depth_video(p):
                continue
            vids.append((day, p))
    return vids

def load_mask(path):
    p = Path(path)
    if not p.exists(): return None
    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    return (m>127).astype(np.uint8)

def apply_mask_rgb(img, mask):
    if mask is None: return img
    H,W = img.shape[:2]
    if mask.shape != (H,W):
        mask_r = cv2.resize(mask, (W,H), interpolation=cv2.INTER_NEAREST)
    else:
        mask_r = mask
    out = img.copy()
    out[mask_r==0] = 0
    return out

def polygon_contains(poly, pt):
    return cv2.pointPolygonTest(np.array(poly, np.int32), (float(pt[0]), float(pt[1])), False) >= 0

def dilate_polygon(poly, px=5):
    cnt = np.array(poly, np.int32).reshape(-1,1,2)
    x,y,w,h = cv2.boundingRect(cnt)
    roi = np.zeros((h+2*px, w+2*px), np.uint8)
    cnt2 = cnt.copy(); cnt2[:,:,0]-=x; cnt2[:,:,1]-=y
    cnt2[:,:,0]+=px;  cnt2[:,:,1]+=px
    cv2.fillPoly(roi, [cnt2], 255)
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*px+1, 2*px+1))
    roi = cv2.dilate(roi, ker, iterations=1)
    ys, xs = np.where(roi>0)
    if len(xs)<3: return poly
    pts = np.stack([xs, ys], axis=1).astype(np.int32)
    pts[:,0]+=x-px; pts[:,1]+=y-px
    hull = cv2.convexHull(pts)
    return hull.reshape(-1,2).tolist()

def load_gray(path: Path):
    if not path.exists(): return None
    im = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return im

def get_background_for_video(vpath: Path):

    if USE_LOCAL_BG:
        local_bg = vpath.parent / LOCAL_BG_NAME
        im = load_gray(local_bg)
        if im is not None:
            return im
    return None

def read_time_txt_if_any(vpath, nframes):
    if not USE_TIME_TXT: return None
    cand1 = vpath.with_suffix(".time.txt")
    cand2 = vpath.parent / "time.txt"
    for tf in [cand1, cand2]:
        if tf.exists():
            ts=[]
            with open(tf,"r") as f:
                for line in f:
                    line=line.strip()
                    if not line: continue
                    try: ts.append(float(line))
                    except: pass
            if len(ts)>=nframes:
                return ts
    return None

# ROI loaders
def _load_roi_simple(json_path):
    with open(json_path,"r") as f:
        data=json.load(f)
    rois={}
    for key in ["feeder","drinker","toy"]:
        poly = data.get(key, None)
        if isinstance(poly, list) and len(poly)>=3:
            rois[key]=poly
    return rois

def _load_roi_coco(json_path):
    with open(json_path,"r") as f:
        coco=json.load(f)
    if not all(k in coco for k in ["images","annotations","categories"]):
        return {}
    id2name = {c["id"]: c.get("name","") for c in coco.get("categories",[])}
    wanted  = {"feeder","drinker","toy"}
    polys   = defaultdict(list)
    for ann in coco.get("annotations", []):
        cat_name = id2name.get(ann.get("category_id"), "")
        if cat_name not in wanted:
            continue
        seg = ann.get("segmentation", [])
        if seg:
            pts = seg[0] if isinstance(seg[0], list) else seg
            poly = [[float(pts[i]), float(pts[i+1])] for i in range(0,len(pts),2)]
            if len(poly) >= 3:
                polys[cat_name].append(poly)
        else:
            x,y,w,h = ann.get("bbox",[0,0,0,0])
            rect = [[x,y],[x+w,y],[x+w,y+h],[x,y+h]]
            polys[cat_name].append(rect)
    rois={}
    for k, arr in polys.items():
        if len(arr)==1:
            rois[k]=arr[0]
        else:
            areas=[]
            for poly in arr:
                cnt=np.array(poly,np.float32).reshape(-1,1,2)
                areas.append(abs(cv2.contourArea(cnt)))
            rois[k]=arr[int(np.argmax(areas))]
    return rois

def load_rois_auto(roi_path: Path):
    if not roi_path.exists():
        pass
        return {}
    with open(roi_path,"r") as f:
        obj=json.load(f)
    if all(k in obj for k in ["images","annotations","categories"]):
        rois = _load_roi_coco(roi_path)
        print("[INFO] ROI from COCO. Keys:", list(rois.keys()))
        return rois
    else:
        rois = _load_roi_simple(roi_path)
        print("[INFO] ROI from JSON. Keys:", list(rois.keys()))
        return rois

# perceptual hash (aHash)
def ahash_gray(img, size=8):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if (len(img.shape)==3 and img.shape[2]==3) else img
    g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
    m = g.mean()
    bits = (g > m).astype(np.uint8).flatten()
    return ''.join([f'{int("".join(map(str,bits[i:i+4])), 2):x}' for i in range(0, size*size, 4)])  # 64bit hex

def hamming_hex(h1, h2):
    return (int(h1,16) ^ int(h2,16)).bit_count()


In [ ]:
def build_candidates_csv():
    mask        = load_mask(GLOBAL_MASK_PATH)
    rois        = load_rois_auto(ROI_PATH)
    rois_dilated= {k: dilate_polygon(v, ROI_DILATE_PX) for k,v in rois.items()}

    done_pairs = set()
    header_needed = True
    sleepy_kept_so_far = 0
    if CANDIDATES_CSV.exists():
        try:
            df_done = pd.read_csv(CANDIDATES_CSV)
            if {"day","video"}.issubset(df_done.columns):
                done_pairs = set(map(tuple, df_done[["day","video"]].drop_duplicates().values.tolist()))
            header_needed = False
            if "motion_level" in df_done.columns:
                if "near_roi_flag" in df_done.columns:
                    sleepy_kept_so_far = int(
                        ((df_done["motion_level"].astype(str)=="low") &
                         (df_done["near_roi_flag"].fillna(0).astype(int)==0)).sum()
                    )
                else:
                    sleepy_kept_so_far = int(
                        ((df_done["motion_level"].astype(str)=="low") &
                         (df_done.get("roi_name","").astype(str).str.len()==0)).sum()
                    )
            print(f"[RESUME] sleepy already in CSV: {sleepy_kept_so_far}")
        except Exception as e:
            print("[WARN] Cannot read existing CSV, recreating:", e)
            CANDIDATES_CSV.unlink(missing_ok=True)

    f = open(CANDIDATES_CSV, "a", newline="")
    wr = csv.writer(f)
    if header_needed:
        wr.writerow([
            "group_id","day","video","center_frame","center_ts","frames",
            "trigger_type","roi_name","speed","d_aspect","aspect","blur_var",
            "fg_area","motion_level","near_roi_flag"
        ])

    vids = list_color_videos()
    print(f"[INFO] COLOR videos:", len(vids))

    CLIP_TEST_SAMPLES_SEC   = 20.0
    CLIP_SAMPLE_STEP_SEC    = 2.0
    CLIP_FG_AREA_THRESH     = FG_AREA_MIN
    CLIP_SPEED_RMS_THRESH   = SPEED_RMS_LOW
    CLIP_SLEEPY_MAX_BURSTS  = 1
    CLIP_SLEEPY_STRICT      = True

    for day, vpath in tqdm(vids):
        pair = (day, str(vpath))
        if pair in done_pairs:
            continue

        cap = cv2.VideoCapture(str(vpath))
        if not cap.isOpened():
            print("[WARN] cannot open:", vpath)
            continue

        fps = cap.get(cv2.CAP_PROP_FPS) or FPS_FALLBACK
        nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        ts_list = read_time_txt_if_any(vpath, nframes)

        bg_local = get_background_for_video(vpath)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ok0, first_frame = cap.read()
        if not ok0:
            cap.release()
            print("[WARN] empty/broken:", vpath)
            continue
        first_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
        mask_gray0 = apply_mask_rgb(cv2.cvtColor(first_gray, cv2.COLOR_GRAY2BGR), mask)
        first_gray = cv2.cvtColor(mask_gray0, cv2.COLOR_BGR2GRAY)

        clip_is_sleepy = False
        if bg_local is None:
            test_T = min(CLIP_TEST_SAMPLES_SEC, nframes / max(fps,1e-6))
            steps  = max(1, int(test_T // CLIP_SAMPLE_STEP_SEC))
            fg_areas = []
            speeds   = []
            prev_centroid = None

            for s in range(steps):
                t = int(min(nframes-1, (s * CLIP_SAMPLE_STEP_SEC) * fps))
                cap.set(cv2.CAP_PROP_POS_FRAMES, t)
                ok, fr = cap.read()
                if not ok:
                    break

                g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
                g = cv2.cvtColor(apply_mask_rgb(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR), mask),
                                 cv2.COLOR_BGR2GRAY)

                fg = cv2.absdiff(g, first_gray)
                th = max(10, int(np.mean(fg) + np.std(fg)))
                _, fg_bin = cv2.threshold(fg, th, 255, cv2.THRESH_BINARY)
                fg_bin = cv2.medianBlur(fg_bin, 5)
                if mask is not None and mask.shape == fg_bin.shape:
                    fg_bin = cv2.bitwise_and(fg_bin, fg_bin, mask=mask*255)

                H,W = g.shape[:2]
                fg_area = float((fg_bin>0).sum()) / float(H*W + 1e-6)
                fg_areas.append(fg_area)

                # centroid + speed
                cnts,_ = cv2.findContours(fg_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                centroid=None
                if cnts:
                    c = max(cnts, key=cv2.contourArea)
                    x,y,w,h = cv2.boundingRect(c)
                    if w>0 and h>0:
                        centroid=(x+w/2.0, y+h/2.0)
                if centroid and prev_centroid:
                    dt = CLIP_SAMPLE_STEP_SEC
                    dx = centroid[0]-prev_centroid[0]
                    dy = centroid[1]-prev_centroid[1]
                    speed = (dx**2+dy**2)**0.5 / max(dt,1e-6)
                else:
                    speed = 0.0
                speeds.append(speed)
                prev_centroid = centroid

            fg_mean = float(np.mean(fg_areas)) if fg_areas else 0.0
            sp_rms  = (np.mean(np.square(speeds))**0.5) if speeds else 0.0

            clip_is_sleepy = (fg_mean < CLIP_FG_AREA_THRESH) and (sp_rms < CLIP_SPEED_RMS_THRESH)
            if clip_is_sleepy:
                pass

        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        prev_gray=None
        prev_centroid=None
        speed_hist = deque(maxlen=SPEED_PEAK_WIN)
        aspect_hist= deque(maxlen=3)
        in_roi_state = {}  # roi_name -> inside?
        bucket_activity = defaultdict(list)

        recent_hashes = []   # (ts_ms, hashhex)
        last_keep_ts  = -1e9

        idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray = cv2.cvtColor(apply_mask_rgb(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), mask),
                                cv2.COLOR_BGR2GRAY)

            if bg_local is not None and bg_local.shape == gray.shape:
                fg = cv2.absdiff(gray, bg_local)
            else:
                fg = cv2.absdiff(gray, first_gray)

            th = max(10, int(np.mean(fg) + np.std(fg)))
            _, fg_bin = cv2.threshold(fg, th, 255, cv2.THRESH_BINARY)
            fg_bin = cv2.medianBlur(fg_bin, 5)
            if mask is not None and mask.shape == fg_bin.shape:
                fg_bin = cv2.bitwise_and(fg_bin, fg_bin, mask=mask*255)

            H, W = gray.shape[:2]
            fg_area = float((fg_bin > 0).sum()) / float(H * W + 1e-6)

            cnts, _ = cv2.findContours(fg_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            centroid=None
            aspect=None
            if cnts:
                c = max(cnts, key=cv2.contourArea)
                x,y,w,h = cv2.boundingRect(c)
                if w>0 and h>0:
                    centroid=(x+w/2.0, y+h/2.0)
                    aspect = h/max(1.0*w, 1.0)

            speed = 0.0
            if centroid and prev_centroid:
                dt = 1.0/(fps if fps>0 else FPS_FALLBACK)
                dx = centroid[0]-prev_centroid[0]
                dy = centroid[1]-prev_centroid[1]
                speed = (dx**2 + dy**2) ** 0.5 / max(dt,1e-6)

            speed_hist.append(speed)
            aspect_hist.append(aspect if aspect is not None else (aspect_hist[-1] if aspect_hist else 0.0))
            d_aspect = (aspect_hist[-1] - aspect_hist[-2]) if len(aspect_hist)>=2 else 0.0
            varlap = float(cv2.Laplacian(gray, cv2.CV_64F).var())

            # near ROI?
            near_roi=False
            roi_name=""
            if rois_dilated and centroid:
                for name, poly in rois_dilated.items():
                    if polygon_contains(poly, centroid):
                        near_roi=True
                        roi_name=name
                        break

            speed_rms = (np.mean(np.square(list(speed_hist))) ** 0.5) if len(speed_hist) > 0 else 0.0
            motion_level = (
                "low" if (speed_rms < SPEED_RMS_LOW and fg_area < FG_AREA_MIN and abs(d_aspect) < ASPECT_DELTA_LOW)
                else "mid" if speed_rms < 6.0
                else "high"
            )
            is_sleepy = (motion_level == "low") and (not near_roi)

            # triggers
            trigger_type = None
            if centroid and rois_dilated:
                inside_now = False
                for nm, poly in rois_dilated.items():
                    if polygon_contains(poly, centroid):
                        inside_now=True
                        break
                prev_inside = in_roi_state.get(roi_name, False)
                if inside_now and not prev_inside:
                    trigger_type="roi_enter"
                    in_roi_state[roi_name]=True
                elif (not inside_now) and prev_inside:
                    trigger_type="roi_exit"
                    in_roi_state[roi_name]=False

            if trigger_type is None and abs(d_aspect) >= ASPECT_DELTA_TH:
                trigger_type = "onset" if d_aspect>0 else "offset"

            if trigger_type is None and len(speed_hist)==SPEED_PEAK_WIN:
                mid = speed_hist[len(speed_hist)//2]
                if mid == max(speed_hist) and mid > 1.0:
                    trigger_type="speed_peak"

            ts = (ts_list[idx] if (ts_list and idx < len(ts_list))
                  else (idx/(fps if fps>0 else FPS_FALLBACK)))

            if trigger_type and varlap >= BLUR_MIN_VARLAP:
                center_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
                ah = ahash_gray(center_img)
                ms = int(ts*1000)

                # aHash window
                recent_hashes = [
                    (t,h) for (t,h) in recent_hashes
                    if abs(ms - t) <= P1_HASH_WINDOW_S*1000
                ]
                too_similar = any(hamming_hex(ah, h) <= P1_AHAMMING_SKIP for (t,h) in recent_hashes)
                too_close   = (ts - last_keep_ts) < P1_MIN_GAP_GLOBAL_S

                if not (too_similar or too_close):
                    recent_hashes.append((ms, ah))

                    if clip_is_sleepy and CLIP_SLEEPY_STRICT:
                        if (not near_roi) or (motion_level == "low"):
                            pass
                        else:
                            bucket = int(ts)
                            bucket_activity[bucket].append({
                                "i": idx, "ts": ts, "trigger": trigger_type, "roi": roi_name,
                                "speed": float(speed),
                                "aspect": float(aspect_hist[-1] if len(aspect_hist) else 0.0),
                                "d_aspect": float(d_aspect),
                                "blur": float(varlap),
                                "fg_area": float(fg_area),
                                "motion_level": motion_level,
                                "near_roi_flag": int(near_roi),
                                "is_sleepy": bool(is_sleepy)
                            })
                            last_keep_ts = ts
                    else:
                        bucket = int(ts)
                        bucket_activity[bucket].append({
                            "i": idx, "ts": ts, "trigger": trigger_type, "roi": roi_name,
                            "speed": float(speed),
                            "aspect": float(aspect_hist[-1] if len(aspect_hist) else 0.0),
                            "d_aspect": float(d_aspect),
                            "blur": float(varlap),
                            "fg_area": float(fg_area),
                            "motion_level": motion_level,
                            "near_roi_flag": int(near_roi),
                            "is_sleepy": bool(is_sleepy)
                        })
                        last_keep_ts = ts

            prev_gray = gray
            prev_centroid = centroid
            idx += 1

        cap.release()

        chosen=[]
        for sec in sorted(bucket_activity.keys()):
            lst = bucket_activity[sec]
            lst.sort(
                key=lambda r: (
                    r["trigger"] in ("roi_enter","roi_exit"),
                    r["speed"],
                    abs(r["d_aspect"])
                ),
                reverse=True
            )
            chosen.extend(lst[:TOPK_PER_SEC])

        used_by_type = defaultdict(int)
        used_per_min = defaultdict(int)
        used_sleep_m = defaultdict(int)
        centers=[]
        last_ts2=-1e9
        per_clip_used=0

        for r in sorted(chosen, key=lambda x: x["ts"]):
            minute = int(r["ts"]//60)
            ttype  = r["trigger"]

            if used_by_type[(minute, ttype)] >= CAP_BURST_PER_MIN_PER_TYPE.get(ttype, 9999):
                continue
            if used_per_min[minute] >= CAP_BURST_PER_MIN_TOTAL:
                continue
            if (r["ts"] - last_ts2) < MIN_GAP_BURST_S:
                continue
            if r.get("is_sleepy", False) and used_sleep_m[minute] >= CAP_SLEEP_PER_MINUTE:
                continue
            if clip_is_sleepy and per_clip_used >= CLIP_SLEEPY_MAX_BURSTS:
                continue

            used_by_type[(minute, ttype)] += 1
            used_per_min[minute] += 1
            used_sleep_m[minute] += int(r.get("is_sleepy", False))
            centers.append(r)
            last_ts2 = r["ts"]
            per_clip_used += 1

        for c in centers:
            if c.get("is_sleepy", False) and sleepy_kept_so_far >= SLEEPY_KEEP_QUOTA:
                continue

            t = int(c["i"])
            offs = [o * BURST_STRIDE_FRAMES for o in BURST_OFFSETS]
            cand = [t + o for o in offs]

            # clamp & pad
            shift = 0
            if cand[0] < 0:
                shift = -cand[0]
            elif cand[-1] > (nframes - 1):
                shift = (nframes - 1) - cand[-1]
            cand = [min(max(x + shift, 0), nframes - 1) for x in cand]

            fr_idx, seen = [], set()
            for x in cand:
                if x not in seen:
                    fr_idx.append(x)
                    seen.add(x)
            while len(fr_idx) < BURST_LEN:
                fr_idx.append(fr_idx[-1])

            group_id = f"{day}/{vpath.stem}/{int(c['ts']*1000)}"
            wr.writerow([
                group_id, day, str(vpath), t, round(c["ts"],3),
                "|".join(map(str, fr_idx)),
                c["trigger"], c["roi"],
                float(c["speed"]), float(c["d_aspect"]),
                0.0 if c["aspect"] is None else float(c["aspect"]),
                float(c["blur"]),
                float(c.get("fg_area", 0.0)),
                c.get("motion_level",""),
                int(c.get("near_roi_flag", 0))
            ])
            if c.get("is_sleepy", False):
                sleepy_kept_so_far += 1

    f.close()
    print("[DONE][RESUME] candidates:", CANDIDATES_CSV)
    print(f"[INFO] sleepy kept total (global): {sleepy_kept_so_far}/{SLEEPY_KEEP_QUOTA}")

build_candidates_csv()


In [ ]:
import pandas as pd
from pathlib import Path

def thin_candidates_csv_by_gap(
    in_csv: Path = CANDIDATES_CSV,
    out_csv: Path = None,
    min_gap_s: float = 12.0
):
    """
    """
    if out_csv is None:
        out_csv = in_csv.with_name(in_csv.stem + f"_thinned_{int(min_gap_s)}s.csv")

    if not in_csv.exists():
        print("[ERROR] input CSV not found:", in_csv)
        return

    df = pd.read_csv(in_csv)
    if len(df) == 0:
        print("[WARN] input CSV empty:", in_csv)
        return

    if not {"video", "center_ts"}.issubset(df.columns):
        pass
        return

    print(f"[THIN-CVS] input: {in_csv}  rows={len(df)}  min_gap_s={min_gap_s}")

    # sort theo video, time
    df = df.sort_values(["video", "center_ts"]).reset_index(drop=True)

    kept_rows = []
    last_keep_ts_by_video = {}

    for idx, row in df.iterrows():
        v  = row["video"]
        ts = float(row["center_ts"])
        last_ts = last_keep_ts_by_video.get(v, -1e9)
        if ts - last_ts >= min_gap_s:
            kept_rows.append(row)
            last_keep_ts_by_video[v] = ts

    df_thin = pd.DataFrame(kept_rows).reset_index(drop=True)
    df_thin.to_csv(out_csv, index=False)

    pass
    return out_csv

new_csv = thin_candidates_csv_by_gap(
    in_csv=CANDIDATES_CSV,
    min_gap_s=20.0
)
print("New thinned CSV:", new_csv)


In [ ]:
def _balanced_sample_bursts(df, target_bursts, seed=42):
    df = df.copy()
    df["near_roi_flag"] = df.get("near_roi_flag", 0)
    df["near_roi"] = df["near_roi_flag"].apply(lambda v: "yes" if int(v)==1 else "no")
    buckets = []
    for t in sorted(df["trigger_type"].unique()):
        for nr in ["yes","no"]:
            sub = df[(df["trigger_type"]==t) & (df["near_roi"]==nr)]
            if len(sub)>0:
                buckets.append(sub.sample(frac=1.0, random_state=seed))
    if not buckets:
        return df.sample(n=min(target_bursts, len(df)), random_state=seed)
    k = len(buckets)
    per = max(1, target_bursts // k)
    selected = []; remain = target_bursts
    for b in buckets:
        take = min(per, len(b))
        selected.append(b.iloc[:take]); remain -= take
    sel = pd.concat(selected, ignore_index=True)
    if remain > 0 and len(df) > len(sel):
        rest = df.drop(sel.index, errors="ignore")
        add = rest.sample(n=min(remain, len(rest)), random_state=seed)
        sel = pd.concat([sel, add], ignore_index=True)
    if len(sel) > target_bursts:
        sel = sel.sample(n=target_bursts, random_state=seed)
    return sel

def _make_strided_window(center, nframes, burst_len, stride):
    offs = [o * stride for o in [-3,-2,-1,0,1,2]]
    cand = [center + o for o in offs]
    shift = 0
    if cand[0] < 0: shift = -cand[0]
    elif cand[-1] > (nframes - 1): shift = (nframes - 1) - cand[-1]
    cand = [min(max(x + shift, 0), nframes - 1) for x in cand]
    while len(cand) < burst_len:
        cand.append(cand[-1])
    return cand[:burst_len]

def sample_and_export_from_csv():
    if not CANDIDATES_CSV.exists():
        print("[ERROR] Candidates CSV not found:", CANDIDATES_CSV); return
    df_all = pd.read_csv(CANDIDATES_CSV)
    if len(df_all) == 0:
        print("[WARN] empty candidates."); return

    df = df_all[~df_all["video"].str.lower().str.contains("depth")].copy()
    key_cols=["day","video","center_frame","center_ts","frames","trigger_type","roi_name"]
    have=[c for c in key_cols if c in df.columns]
    if have:
        before=len(df); df=df.drop_duplicates(subset=have, keep="first").reset_index(drop=True); after=len(df)
        if after<before: print(f"[DEDUP] dropped {before-after} duplicate rows")

    target_bursts = math.ceil(TARGET_IMAGES / BURST_LEN)
    print(f"[PLAN] Need ~{target_bursts} bursts (~{TARGET_IMAGES} images). Candidates: {len(df)}")

    if "motion_level" not in df.columns: df["motion_level"]="unknown"
    if "near_roi_flag" not in df.columns: df["near_roi_flag"] = (df.get("roi_name","").astype(str).str.len()>0).astype(int)
    sleep_mask = (df["motion_level"].astype(str)=="low") & (df["near_roi_flag"].astype(int)==0)

    if target_bursts >= len(df):
        non_sleep = df[~sleep_mask].copy(); sleepy_df = df[sleep_mask].copy()
        if SLEEP_STRICT:
            sel_bursts = non_sleep
            print(f"[SELECT][STRICT] ONLY non-sleepy = {len(sel_bursts)} (drop sleepy: {len(sleepy_df)})")
        else:
            max_sleep = int(MAX_SLEEP_RATIO * len(df))
            take_sleep = min(max_sleep, len(sleepy_df))
            sleepy_add = sleepy_df.sample(n=take_sleep, random_state=RANDOM_SEED) if take_sleep>0 else sleepy_df.iloc[:0]
            sel_bursts = pd.concat([non_sleep, sleepy_add], ignore_index=True)
            pass
    else:
        base = _balanced_sample_bursts(df, target_bursts, seed=RANDOM_SEED) if SAMPLER_MODE=="balanced" else df.sample(n=target_bursts, random_state=RANDOM_SEED)
        if SLEEP_STRICT:
            base = base[~sleep_mask]
            rest_non_sleep = df[~sleep_mask].drop(index=base.index, errors="ignore")
            need = target_bursts - len(base)
            if need>0 and len(rest_non_sleep)>0:
                add = rest_non_sleep.sample(n=min(need, len(rest_non_sleep)), random_state=RANDOM_SEED)
                base = pd.concat([base, add], ignore_index=True)
        else:
            sleepy_sel = base[sleep_mask]; cap = int(MAX_SLEEP_RATIO * len(base))
            if len(sleepy_sel) > cap:
                drop_ids = sleepy_sel.sample(n=len(sleepy_sel)-cap, random_state=RANDOM_SEED).index
                base = base.drop(index=drop_ids)
                rest = df.drop(index=base.index, errors="ignore"); rest = rest[~sleep_mask]
                need = target_bursts - len(base)
                if need>0 and len(rest)>0:
                    add = rest.sample(n=min(need, len(rest)), random_state=RANDOM_SEED)
                    base = pd.concat([base, add], ignore_index=True)
        sel_bursts = base

    sel_bursts = sel_bursts.sort_values(["video","center_ts"]).reset_index(drop=True)
    kept=[]; last_keep_ts_by_video={}
    for _, r in sel_bursts.iterrows():
        v=r["video"]; ts=float(r["center_ts"])
        if (ts - last_keep_ts_by_video.get(v,-1e9)) >= P2_MIN_GAP_GLOBAL_S:
            kept.append(r); last_keep_ts_by_video[v]=ts
    sel_bursts=pd.DataFrame(kept)
    pass

    if BATCH_BURSTS is not None:
        sel_bursts = sel_bursts.head(BATCH_BURSTS)
        print(f"[BATCH] exporting first {len(sel_bursts)} bursts")

    # export + hash guard
    msk = load_mask(GLOBAL_MASK_PATH)
    with open(MANIFEST, "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["group_id","order","img_path","day","video","center_frame","center_ts","trigger_type","roi_name","near_roi"])

        exported_imgs=0; exported_bursts=0
        recent_hashes = defaultdict(list)  # video -> list of (ts_ms, ahash)

        groups = list(sel_bursts.groupby(["day","video"], sort=False))
        for (day, video), g in tqdm(groups, total=len(groups)):
            vpath = Path(video)
            cap = cv2.VideoCapture(str(vpath))
            if not cap.isOpened(): print("[WARN] cannot open:", vpath); continue
            nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
            vid_hash = hashlib.md5(str(video).encode()).hexdigest()[:8]
            vid_stem = Path(video).stem

            for _, row in g.iterrows():
                group_id=row["group_id"]; center_i=int(row["center_frame"]); center_ts=float(row["center_ts"])
                trigger=row["trigger_type"]; roi_name=row["roi_name"] if isinstance(row["roi_name"],str) else ""
                near_roi = 1 if (int(row.get("near_roi_flag",0))==1 or (isinstance(roi_name,str) and len(roi_name)>0)) else 0
                ms = int(center_ts*1000)

                cap.set(cv2.CAP_PROP_POS_FRAMES, center_i)
                okc, center_frame = cap.read()
                if not okc: continue
                center_frame_m = apply_mask_rgb(center_frame, msk)
                ah = ahash_gray(center_frame_m)
                recent_hashes[video] = [(t,h) for (t,h) in recent_hashes[video] if abs(ms - t) <= P2_HASH_WINDOW_S*1000]
                if any(hamming_hex(ah, h) <= P2_AHAMMING_SKIP for (t,h) in recent_hashes[video]):
                    continue
                recent_hashes[video].append((ms, ah))

                # burst frames
                frames = [int(x) for x in str(row["frames"]).split("|") if str(x).strip().isdigit()]
                if len(frames)!=BURST_LEN:
                    frames = _make_strided_window(center_i, nframes, BURST_LEN, BURST_STRIDE_FRAMES)
                frames = [min(max(fi,0), nframes-1) for fi in sorted(frames)]

                for order, fi in enumerate(frames):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
                    ok, frame = cap.read()
                    if not ok:
                        alt=min(max(fi+1,0), nframes-1); cap.set(cv2.CAP_PROP_POS_FRAMES, alt)
                        ok, frame = cap.read()
                        if not ok: continue
                    frame_m = apply_mask_rgb(frame, msk)
                    out_name = f"burst_{vid_stem}_{vid_hash}_{ms}_f{fi}_k{order}.jpg"
                    out_path = OUT_IMG_DIR / out_name
                    if out_path.exists():
                        continue
                    if not cv2.imwrite(str(out_path), frame_m):
                        print("[WARN] write failed:", out_path); continue

                    exported_imgs += 1
                    wr.writerow([group_id, order, str(out_path), day, str(video), center_i, center_ts, trigger, roi_name, near_roi])
                exported_bursts += 1
            cap.release()

    n_imgs_on_disk = sum(1 for e in os.scandir(OUT_IMG_DIR) if e.is_file())
    pass
    print(f"[DISK] Images now in folder: ~{n_imgs_on_disk}")

sample_and_export_from_csv()


In [ ]:
import os

PARENT_DIR = "/content/drive/MyDrive/pig-selected_frame_attribute/images_frame_attribute"
CHILD_DIR  = "/content/drive/MyDrive/pig-selected_frame_attribute/images_frame_attribute/images_frame_attribute_(2)_1"

assert os.path.isdir(PARENT_DIR)
assert os.path.isdir(CHILD_DIR)
print("OK path!")


In [ ]:
import os, shutil
from pathlib import Path

DRY_RUN = False

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}

moved, skipped = 0, 0
renamed = 0
to_move = []

for name in os.listdir(CHILD_DIR):
    src = os.path.join(CHILD_DIR, name)
    if not os.path.isfile(src):
        continue
    if Path(name).suffix.lower() not in IMG_EXTS:
        skipped += 1
        continue
    to_move.append(src)

pass

def unique_target_path(dst_dir, filename):
    p = Path(dst_dir) / filename
    if not p.exists():
        return str(p), False
    stem, suf = p.stem, p.suffix
    k = 1
    while True:
        candidate = Path(dst_dir) / f"{stem}_{k:03d}{suf}"
        if not candidate.exists():
            return str(candidate), True
        k += 1

for src in to_move:
    base = os.path.basename(src)
    dst, renamed_flag = unique_target_path(PARENT_DIR, base)
    if not DRY_RUN:
        shutil.move(src, dst)
    moved += 1
    if renamed_flag:
        renamed += 1

pass
pass
